# 04 — Idiot Sandwich: TRL DPOTrainer + LoRA persona tuning (Task 5)

Accelerator: **GPU T4 x2** (one GPU is used), Internet **on**. No inputs needed. Expected time: ~10–15 min.

Trains a LoRA adapter on `HuggingFaceTB/SmolLM2-360M-Instruct` with TRL's `DPOTrainer` on
`data/persona_lora/ramsay_style_preferences.json` (200 train / 20 held-out pairs), then generates
responses from the base model and the LoRA-DPO model on the fixed style-evaluation prompts
(`eval_sets/persona_style_eval.json`: cooking, coding, general help, safety). When done: **Save Version**.

In [ ]:
REPO_URL = "https://github.com/madgod2004/small-model-alignment-starter.git"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"
BRANCH = "main"

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.11.0" "trl>=0.12.0" pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
!python scripts/check_env.py --expect_gpus 2

## 1. Train the LoRA adapter with TRL DPOTrainer

In [ ]:
!mkdir -p outputs/logs
!PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 python scripts/train_lora_dpo.py --config configs/lora_dpo.yaml 2>&1 | grep -v -i warn | tee outputs/logs/train_lora_dpo.log

In [ ]:
import json
import matplotlib.pyplot as plt

summary = json.load(open("outputs/lora_dpo/summary.json"))
print(f"trainable {summary['trainable_parameters']:,} / {summary['total_parameters']:,} ({summary['trainable_percent']:.2f}%), "
      f"{summary['optimizer_steps']} steps, {summary['train_seconds']:.0f}s")
for name in ["initial_eval", "final_eval"]:
    e = summary[name]
    print(name, {k: round(e[k], 4) for k in ["eval_loss", "eval_rewards/accuracies", "eval_rewards/margins"] if k in e})

rows = [json.loads(line) for line in open("outputs/lora_dpo/metrics.jsonl")]
train = [r for r in rows if "loss" in r and "eval_loss" not in r and "step" in r]
evals = [r for r in rows if "eval_loss" in r and "step" in r]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, key, label in zip(axes, ["loss", "rewards/accuracies", "rewards/margins"], ["DPO loss", "reward accuracy", "reward margin"]):
    ax.plot([r["step"] for r in train], [r[key] for r in train], alpha=0.6, label="train")
    ax.plot([r["step"] for r in evals], [r["eval_" + key] for r in evals], "o-", label="held-out eval")
    ax.set_xlabel("optimizer step")
    ax.set_title(f"LoRA-DPO persona: {label}")
    ax.legend()
plt.tight_layout()
plt.show()

## 2. Generate on the style-evaluation prompts: base SmolLM2 vs LoRA-DPO
Same decoding settings (`configs/generation.yaml`) with SmolLM2's own chat template.

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --generator smollm2-360m-instruct --eval_set eval_sets/persona_style_eval.json --output_path outputs/generations/smollm_base_persona_eval.json --prompt_format chat_template --overwrite 2>&1 | grep -v -i warn
!python scripts/generate.py --model_path outputs/lora_dpo/merged --generator smollm2-360m-lora-dpo --eval_set eval_sets/persona_style_eval.json --output_path outputs/generations/lora_dpo_persona_eval.json --prompt_format chat_template --overwrite 2>&1 | grep -v -i warn

In [ ]:
base = json.load(open("outputs/generations/smollm_base_persona_eval.json"))
lora = json.load(open("outputs/generations/lora_dpo_persona_eval.json"))
for b, l in zip(base, lora):
    print("=" * 100)
    print(f"[{b['category']}] {b['instruction']}")
    print("--- BASE:", b["output"][:600])
    print("--- LORA:", l["output"][:600])

## 3. Archive
The zip holds metrics, summary, split, logs, and both generation files (the merged model stays in the saved output).

In [ ]:
!zip -qr /kaggle/working/lora_dpo_outputs.zip outputs/lora_dpo/metrics.jsonl outputs/lora_dpo/summary.json outputs/lora_dpo/config.json outputs/lora_dpo/split.json outputs/logs/train_lora_dpo.log outputs/generations/smollm_base_persona_eval*.json outputs/generations/lora_dpo_persona_eval*.json
!ls -lh /kaggle/working/*.zip && du -sh outputs/lora_dpo/adapter outputs/lora_dpo/merged